# Understanding LeNet-5: The Grandfather of Convolutional Neural Networks

Welcome to this tutorial! Today, we are traveling back to 1998 to build **LeNet-5**, one of the earliest and most influential Convolutional Neural Networks (CNNs). Designed by Yann LeCun, this architecture was originally built to recognize handwritten zip codes on checks for the US Postal Service.

Since this is an architectural deep dive, we will skip complex feature engineering and build this network from scratch using TensorFlow and Keras, training it on the famous MNIST dataset of handwritten digits. Let's dive in!

In [1]:
import tensorflow
from tensorflow import keras
from keras.layers import Dense,Convolution2D,Flatten,AveragePooling2D
from keras import Sequential
from keras.datasets import mnist

In [2]:
(X_train,y_train),(X_test,y_test)=mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


### Preparing the Data: Why are we padding?

Before feeding our data into the model, we need to make a slight adjustment. The MNIST dataset consists of images that are $28 \times 28$ pixels. However, the original LeNet-5 architecture was specifically designed to take $32 \times 32$ pixel images as input.

To fix this, we use `numpy.pad` to add 2 rows/columns of zeros (black pixels) to all sides of our images.

* **Padding:** Converts our $28 \times 28$ images to $32 \times 32$.
* **Normalization (`/ 255`):** Neural networks prefer small numbers. Dividing by 255 scales our pixel values from a range of $0-255$ down to $0-1$, helping the model learn much faster.
* **One-Hot Encoding:** We convert our target labels (digits 0-9) into categorical formats so our network can output probabilities for each distinct class.

In [5]:
import numpy as np

X_train = X_train.reshape(X_train.shape[0],28,28,1)
X_test = X_test.reshape(X_test.shape[0],28,28,1)

X_train_padded = np.pad(X_train,((0,0),(2,2),(2,2),(0,0)),'constant')
X_test_padded = np.pad(X_test,((0,0),(2,2),(2,2),(0,0)),'constant')

X_train_padded = X_train_padded.astype('float32')/255
X_test_padded = X_test_padded.astype('float32')/255

y_train = keras.utils.to_categorical(y_train,10)
y_test = keras.utils.to_categorical(y_test,10)

### Building LeNet-5: A Trip Back to 1998

Now for the fun part—building the network. If you look closely at the code below, you'll notice a few design choices that might seem a bit old-school today.

**The Historical Fact: Why `tanh`?**
In modern deep learning, the `ReLU` (Rectified Linear Unit) activation function is the undisputed king. But back in 1998, `ReLU` wasn't the standard yet. The most advanced activation functions of the time were `Sigmoid` and `tanh` (Hyperbolic Tangent). We are using `tanh` here to stay true to the classic architecture!

**Layer Breakdown:**
1. **Conv2D (Layer 1):** We start with 6 filters of size $5 \times 5$. These act like tiny flashlights scanning the image to find basic features like edges and curves.
2. **AveragePooling2D (Layer 2):** Why do we use a `pool_size` of $(2,2)$ and a stride of 2? Pooling's job is to shrink our image dimensions by half (from $28 \times 28$ down to $14 \times 14$). By taking the "average" of a $2 \times 2$ pixel grid, we reduce computational load and make the network "spatially invariant" meaning it can still recognize a number even if it's drawn slightly off-center. (Note: Modern networks usually prefer *Max Pooling*, but LeNet originally used *Average Pooling*!).
3. **Conv2D (Layer 3):** We increase to 16 filters, again using a $5 \times 5$ kernel, allowing the network to combine simple edges into complex shapes.
4. **AveragePooling2D (Layer 4):** We half the dimensions again.
5. **Flatten:** We take our 2D feature maps and flatten them into a single 1D array of 400 numbers. This acts as a bridge between the convolutional feature extractors and our final decision making layers.
6. **Dense Layers (The Classifier):** We pass the data through fully connected layers of 120 nodes, then 84 nodes, and finally 10 nodes. The final layer uses a `softmax` activation to give us a clean percentage probability for each digit (0 through 9).

In [6]:
model = Sequential()

model.add(Convolution2D(6,kernel_size=(5,5),padding='valid',activation='tanh',input_shape=(32,32,1)))
model.add(AveragePooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Convolution2D(16,kernel_size=(5,5),padding='valid',activation='tanh'))
model.add(AveragePooling2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(120,activation='tanh'))
model.add(Dense(84,activation='tanh'))
model.add(Dense(10,activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

### Compiling and Training the Model

To train the network, we need to tell it how to measure its mistakes and how to learn from them.

* **Loss Function:** `categorical_crossentropy` is the standard choice for multi-class classification problems like this.
* **Optimizer:** We use `sgd` (Stochastic Gradient Descent), a classic optimization algorithm that slowly nudges the model's weights in the right direction to minimize the loss.
in the old LeNet-5, the optimizer was SGD.

We will train the model for 20 epochs (full passes over the dataset) using a batch size of 128. Let's watch it learn!

In [7]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='sgd',
    metrics=['accuracy']
)

training = model.fit(X_train_padded,y_train,batch_size=128,epochs=20,validation_data=(X_test_padded,y_test))

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 37s 77ms/step - accuracy: 0.7073 - loss: 1.1493 - val_accuracy: 0.8653 - val_loss: 0.5565
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 78ms/step - accuracy: 0.8767 - loss: 0.4701 - val_accuracy: 0.8973 - val_loss: 0.3843
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 89ms/step - accuracy: 0.8983 - loss: 0.3639 - val_accuracy: 0.9115 - val_loss: 0.3180
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 34s 73ms/step - accuracy: 0.9109 - loss: 0.3117 - val_accuracy: 0.9213 - val_loss: 0.2796
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 36s 77ms/step - accuracy: 0.9199 - loss: 0.2768 - val_accuracy: 0.9291 - val_loss: 0.2502
Epoch 6/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 35s 76ms/step - accuracy: 0.9274 - loss: 0.2501 - val_accuracy: 0.9348 - val_loss: 0.2264
Epoch 7/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 35s 76ms/step - accuracy: 0.9337 - loss: 0.2287 - val_accuracy: 0.9394 - val_loss: 0.2080
Epoch 8/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 40s 74ms/step - accuracy: 0.9386 - loss: 0.2105 - 

### Visualizing the Results: Did our model actually learn?

Training numbers are great, but visualizing them helps us truly understand what happened during those 20 epochs. We are going to plot two graphs using `matplotlib`:

1. **Accuracy Graph:** This shows how our model's accuracy improved over time on both the training data and the unseen validation data.
2. **Loss Graph:** This shows how the model's error decreased.

**IF** the training accuracy keeps going up but the validation accuracy starts going down, that's a sign of **overfitting** (memorizing the data instead of learning patterns).